# SCD_Final · step 4 — the pipeline (YOLO11-seg, fine-tuned)

**blood smear image → segment every cell → crop each cell → classify it → count sickle cells.**

Nothing is trained here. Two checkpoints from the earlier steps are loaded and wired together:

| stage | checkpoint | where it came from |
|---|---|---|
| segment | `final/yolo_seg_finetuned.pt` | step 3, fine-tuned on 55 Cuba smears — F1 0.719, mIoU 0.932 |
| classify | `final/<winner>_best.pt` + `final/best_model.json` | step 1 winner (InceptionV3), test macro-F1 0.999 |

The fine-tuned YOLO *also* predicts circular/elongated. The pipeline ignores that and uses it purely
as a detector — the class comes from the step-1 classifier, which is the model that was actually
benchmarked for classification. That is the whole point of a two-stage pipeline.

## Read this before you read the numbers

- **Cells are cropped from the original-resolution image, not from the 1024 canvas.** The segmenter
  works at 1024×1024; the box is mapped back to source pixels and the crop is taken there, then
  resized to 80×80 — the size step 1 trained on. Cropping off the canvas would hand the classifier
  26-pixel cells on `set2`.
- **The end-to-end classification number is optimistic and cannot be fixed here.** Step 1's Cuba
  crops come from the same erythrocytesIDB dataset as these mask smears, and they carry no smear id,
  so it is impossible to prove the classifier never saw cells from these 12 test smears. It probably
  did. The detection half is clean (step 3 held these smears out); the classification half is not.
- **Only Cuba has masks**, so the held-out set is 12 Cuba smears. No cross-population claim.
- Detection numbers should land on step 3's `yolo-finetuned` row. Tiny differences are the canvas
  being passed in memory here instead of re-read from a quality-95 JPEG.


## 1 · Setup

With no configuration, the notebook discovers the checkout from the current working directory,
inventories every `*.pt` checkpoint in `final/`, reads model metadata from `final/best_model.json`,
and reads `data/SCD_Final.zip`. The two-stage pipeline assigns the compatible checkpoint to each
role: `yolo_seg_finetuned.pt` for segmentation and the winner named by the JSON for classification.

Extracted data defaults to `data/scd_final_data/` and results to `results/step4_yolo/`. Every path
can still be overridden with the `SCD_*` environment variables used below.


In [ ]:
import os, re, json, glob, time, zipfile, random
from pathlib import Path
import numpy as np, pandas as pd, cv2, torch, torch.nn as nn
import torchvision.models as tvm
from torchvision import transforms
import matplotlib.pyplot as plt
from ultralytics import YOLO


def find_project_root():
    """Find the checkout when Jupyter starts in either the repo root or final/."""
    override = os.environ.get('SCD_PROJECT_ROOT')
    candidates = [Path(override).expanduser()] if override else [Path.cwd(), *Path.cwd().parents]
    for root in candidates:
        if (root / 'final').is_dir() and (root / 'data').is_dir():
            return root.resolve()
    raise FileNotFoundError('cannot find project root; start Jupyter in the checkout or set SCD_PROJECT_ROOT')


ROOT = find_project_root()
FINAL = Path(os.environ.get('SCD_CHECKPOINT_DIR', ROOT / 'final')).expanduser().resolve()
ZIP = Path(os.environ.get('SCD_DATA_ZIP', ROOT / 'data' / 'SCD_Final.zip')).expanduser().resolve()
DATA = Path(os.environ.get('SCD_DATA_DIR', ROOT / 'data' / 'scd_final_data')).expanduser().resolve()
OUT = Path(os.environ.get('SCD_OUTPUT_DIR', ROOT / 'results' / 'step4_yolo')).expanduser().resolve()
FIG = OUT / 'figures'
for d in (OUT, FIG): os.makedirs(d, exist_ok=True)

SEED    = 42
WORK    = 1024                 # the segmenter's frame; step 3 used the same one
CROP    = 80                   # step 1's crops are 80x80
PAD     = 0.15                 # crop margin, fraction of the box's long side
CLASSES = ['circular', 'elongated']
IOU_T   = 0.50                 # a detection counts as a hit at mask IoU >= this
N_SHOW  = 4

CHECKPOINTS = {p.name: p for p in sorted(FINAL.glob('*.pt'))}
SEG_CKPT = Path(os.environ.get('SCD_YOLO_FT_CKPT',
                                CHECKPOINTS.get('yolo_seg_finetuned.pt',
                                                FINAL / 'yolo_seg_finetuned.pt'))).expanduser().resolve()
SEG_CONF = float(os.environ.get('SCD_SEG_CONF', 0.25))

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'

assert ZIP.is_file(), f'no data archive at {ZIP} -- set SCD_DATA_ZIP to override it'
assert SEG_CKPT.is_file(), f'no segmenter at {SEG_CKPT} -- set SCD_YOLO_FT_CKPT to override it'
seg = YOLO(str(SEG_CKPT))
print(f'torch {torch.__version__} | {DEV}')
print(f'project root {ROOT}')
print(f'data archive {ZIP}')
print(f'checkpoints ({len(CHECKPOINTS)}): {list(CHECKPOINTS)}')
print(f'segmenter {SEG_CKPT}')


## 2 · The classifier from step 1

Everything comes out of `final/best_model.json` — which architecture won, its input size, its checkpoint,
and the **decision threshold that was tuned on validation and frozen**. Nothing is hard-coded, so a
different step-1 winner needs no edit here.

An SVM head cannot be loaded this way (it is a joblib file, not a state dict), so if step 1's winner
is `mobilenet_svm` or `qsvm` this cell stops and tells you to point `SCD_CLF_MODEL` at a deep model.


In [ ]:
BEST_JSON = Path(os.environ.get('SCD_BEST_MODEL_JSON', FINAL / 'best_model.json')).expanduser().resolve()
assert BEST_JSON.is_file(), f'no model metadata at {BEST_JSON}'
with BEST_JSON.open() as f:
    BEST = json.load(f)
CLF_NAME = os.environ.get('SCD_CLF_MODEL', BEST['winner'])
CLF_SIZE = int(BEST.get('input_size') or 299)
CLF_THR  = float(BEST['threshold'])
wanted = Path(BEST['checkpoint']).name if CLF_NAME == BEST['winner'] else f'{CLF_NAME}_best.pt'
CLF_CKPT = Path(os.environ.get('SCD_CLF_CKPT', CHECKPOINTS.get(wanted, FINAL / wanted)))
CLF_CKPT = CLF_CKPT.expanduser().resolve()
assert CLF_CKPT.is_file(), (f'no classifier checkpoint for {CLF_NAME!r} at {CLF_CKPT}; '
                            f'available checkpoints: {list(CHECKPOINTS)}')

def build_backbone(name):
    """step 1's builder, minus the headless branch -- re-headed to 2 classes"""
    if name == 'inceptionv3':
        m = tvm.inception_v3(weights=None, init_weights=False)
        m.fc = nn.Linear(m.fc.in_features, 2)
        m.AuxLogits.fc = nn.Linear(m.AuxLogits.fc.in_features, 2)
    elif name == 'maxvit_t':    m = tvm.maxvit_t(weights=None)
    elif name == 'mobilenetv3': m = tvm.mobilenet_v3_large(weights=None)
    elif name == 'resnet18':    m = tvm.resnet18(weights=None)
    elif name == 'resnet50':    m = tvm.resnet50(weights=None)
    elif name == 'vgg16':       m = tvm.vgg16(weights=None)
    elif name == 'vgg19':       m = tvm.vgg19(weights=None)
    else: raise ValueError(f'{name!r} is not a deep model -- set SCD_CLF_MODEL to one of the seven')
    if name in ('resnet18', 'resnet50'):
        m.fc = nn.Linear(m.fc.in_features, 2)
    elif name in ('maxvit_t', 'mobilenetv3', 'vgg16', 'vgg19'):
        m.classifier[-1] = nn.Linear(m.classifier[-1].in_features, 2)
    return m


clf = build_backbone(CLF_NAME)
clf.load_state_dict(torch.load(CLF_CKPT, map_location=DEV))
clf = clf.to(DEV).eval()

# identical to step 1's TF_EVAL
TF = transforms.Compose([transforms.ToTensor(),
                         transforms.Resize((CLF_SIZE, CLF_SIZE), antialias=True),
                         transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
print(f'classifier {CLF_NAME} @ {CLF_SIZE}px | threshold {CLF_THR:.4f} | {CLF_CKPT}')
print(f"step 1 reported: val macro-F1 {BEST['val_macro_f1']:.4f}, test macro-F1 "
      f"{BEST['test_macro_f1']:.4f}, sickle recall {BEST['sickle_recall']:.4f}")


## 3 · The pipeline

`run_pipeline(path)` is the deliverable. Give it any smear image and it returns one row per detected
cell: mask, box in source pixels, predicted class, P(sickle).

The one detail that matters: **the crop is taken from the source image, not from the canvas.** The
canvas is a square-padded (bottom/right) resize of the source, so `source_px = canvas_px ×
max(H,W) / 1024` with no offset term — that is the only reason the padding goes bottom/right rather
than centred. On a 3136×2352 `set2` smear a cell is ~26 px on the canvas and ~80 px in the source;
the classifier was trained on the latter.


In [ ]:
class Inst:
    """one cell: mask stored cropped to its own box, on the WORK canvas"""
    __slots__ = ('x0', 'y0', 'x1', 'y1', 'm', 'cls', 'score')

    def __init__(self, mask, cls=0, score=1.0):
        ys, xs = np.nonzero(mask)
        self.x0, self.x1 = int(xs.min()), int(xs.max()) + 1
        self.y0, self.y1 = int(ys.min()), int(ys.max()) + 1
        self.m = np.ascontiguousarray(mask[self.y0:self.y1, self.x0:self.x1])
        self.cls, self.score = cls, score

    @property
    def area(self): return int(self.m.sum())

    @property
    def box(self): return [self.x0, self.y0, self.x1, self.y1]

    def full(self, size=None):
        size = size or WORK
        f = np.zeros((size, size), bool)
        f[self.y0:self.y1, self.x0:self.x1] = self.m
        return f


def iou(a, b):
    x0, y0 = max(a.x0, b.x0), max(a.y0, b.y0)
    x1, y1 = min(a.x1, b.x1), min(a.y1, b.y1)
    if x1 <= x0 or y1 <= y0: return 0.0
    inter = np.logical_and(a.m[y0 - a.y0:y1 - a.y0, x0 - a.x0:x1 - a.x0],
                           b.m[y0 - b.y0:y1 - b.y0, x0 - b.x0:x1 - b.x0]).sum()
    return float(inter) / float(a.area + b.area - inter)


def to_canvas(im, interp):
    """square-pad bottom/right, then resize to WORK x WORK"""
    h, w = im.shape[:2]
    s = max(h, w)
    pad = np.zeros((s, s) + im.shape[2:], im.dtype)
    pad[:h, :w] = im
    return cv2.resize(pad, (WORK, WORK), interpolation=interp)


def masks_to_inst(mdata, min_px=30):
    out = []
    a = mdata.cpu().numpy() if torch.is_tensor(mdata) else np.asarray(mdata)
    for m in a:
        m = (m > 0.5).astype(np.uint8)
        if m.shape != (WORK, WORK):
            m = cv2.resize(m, (WORK, WORK), interpolation=cv2.INTER_NEAREST)
        if m.sum() < min_px: continue
        out.append(Inst(m.astype(bool)))
    return out


def segment(canvas):
    r = seg.predict(canvas, imgsz=WORK, conf=SEG_CONF, verbose=False, device=DEV)[0]
    return [] if r.masks is None else masks_to_inst(r.masks.data)


@torch.no_grad()
def classify(crops, bs=64):
    p = []
    for i in range(0, len(crops), bs):
        x = torch.stack([TF(c) for c in crops[i:i + bs]]).to(DEV)
        p.append(torch.softmax(clf(x).float(), 1)[:, 1].cpu().numpy())
    return np.concatenate(p) if p else np.zeros(0)


def run_pipeline(path):
    """smear image -> (instances with cls/score set, canvas, source, per-cell DataFrame)"""
    src = cv2.imread(path)
    assert src is not None, f'cannot read {path}'
    canvas = to_canvas(src, cv2.INTER_AREA)
    insts = segment(canvas)

    s = max(src.shape[:2]) / WORK          # canvas px -> source px; pad is bottom/right, no offset
    crops, boxes, keep = [], [], []
    for k, i in enumerate(insts):
        x0, y0, x1, y1 = [v * s for v in (i.x0, i.y0, i.x1, i.y1)]
        m = PAD * max(x1 - x0, y1 - y0)
        x0, y0 = int(max(0, x0 - m)), int(max(0, y0 - m))
        x1, y1 = int(min(src.shape[1], x1 + m)), int(min(src.shape[0], y1 + m))
        if x1 - x0 < 4 or y1 - y0 < 4: continue        # degenerate box, nothing to classify
        crops.append(cv2.resize(cv2.cvtColor(src[y0:y1, x0:x1], cv2.COLOR_BGR2RGB),
                                (CROP, CROP), interpolation=cv2.INTER_AREA))
        boxes.append((x0, y0, x1, y1)); keep.append(k)

    insts = [insts[k] for k in keep]
    p1 = classify(crops)
    for i, p in zip(insts, p1):
        i.score = float(p); i.cls = int(p >= CLF_THR)
    cells = pd.DataFrame([dict(cell=n, x0=b[0], y0=b[1], x1=b[2], y1=b[3],
                               p_sickle=float(p), pred=CLASSES[int(p >= CLF_THR)])
                          for n, (b, p) in enumerate(zip(boxes, p1))])
    return insts, canvas, src, cells


def report(cells, name=''):
    n = len(cells)
    k = int((cells['pred'] == 'elongated').sum()) if n else 0
    print(f'{name}: {n} cells detected · {k} elongated (sickle) · {100 * k / max(n, 1):.1f}%')
    return dict(image=name, cells=n, sickle=k, sickle_pct=100 * k / max(n, 1))


print('pipeline ready:  insts, canvas, src, cells = run_pipeline(path)')


## 4 · Ground truth for the 12 held-out smears

The split is regenerated with step 3's exact code and seed, so `test` is the same 12 smears step 3
scored — nothing that follows has been seen by the segmenter. Only those 12 get their masks loaded;
the other 67 are listed and skipped.

Mask handling is step 3's: threshold the JPEG-compressed binaries at 127, drop blobs under a quarter
of the smear's median cell, resize `set2`'s quarter-resolution masks up to their source first.


In [ ]:
if not os.path.isdir(f'{DATA}/SCD_Final'):
    print('unzipping...'); zipfile.ZipFile(ZIP).extractall(DATA)
SUPP = f'{DATA}/SCD_Final/_supplementary'

# same enumeration order as step 3: sorted sets, sorted smears, no source.jpg -> dropped
rows = []
for setname in sorted(os.listdir(SUPP)):
    sd = f'{SUPP}/{setname}'
    if not os.path.isdir(sd): continue
    for sm in sorted(os.listdir(sd)):
        p = f'{sd}/{sm}'
        if not os.path.isdir(p): continue
        files = {f.lower(): f for f in os.listdir(p) if not f.startswith('._')}
        if 'source.jpg' not in files: continue
        rows.append(dict(smear=sm, dir=p, src=files['source.jpg'],
                         set='set2' if 'set2' in setname else 'set3'))
meta = pd.DataFrame(rows)

rng = np.random.default_rng(SEED)                    # step 3's split, regenerated
split = {}
for s, part in meta.groupby('set'):
    ids = part.smear.tolist(); rng.shuffle(ids)
    n = len(ids); ntr, nva = int(round(.70 * n)), int(round(.15 * n))
    for i, sm in enumerate(ids):
        split[sm] = 'train' if i < ntr else 'val' if i < ntr + nva else 'test'
meta['split'] = meta['smear'].map(split)
TEST = sorted(meta.loc[meta['split'] == 'test', 'smear'])
print(meta.pivot_table(index='split', columns='set', values='smear', aggfunc='count',
                       margins=True, fill_value=0))


def blobs(mask, cls):
    n, lab, st, _ = cv2.connectedComponentsWithStats((mask > 127).astype(np.uint8), 8)
    a = st[1:, cv2.CC_STAT_AREA]
    if len(a) == 0: return []
    med = float(np.median(a))
    return [Inst(lab == i, cls) for i in range(1, n) if st[i, cv2.CC_STAT_AREA] >= 0.25 * med]


GT, SRC = {}, {}
for r in meta.loc[meta['split'] == 'test'].itertuples():
    src = cv2.imread(f'{r.dir}/{r.src}')
    ins = []
    for ci, c in enumerate(CLASSES):
        f = {x.lower(): x for x in os.listdir(r.dir)}.get(f'mask-{c}.jpg')
        if f is None: continue
        m = cv2.imread(f'{r.dir}/{f}', 0)
        if m.shape[:2] != src.shape[:2]:                       # set2 masks are 1/4 res
            m = cv2.resize(m, (src.shape[1], src.shape[0]), interpolation=cv2.INTER_NEAREST)
        ins += blobs(to_canvas(m, cv2.INTER_NEAREST), ci)
    GT[r.smear] = ins
    SRC[r.smear] = f'{r.dir}/{r.src}'
SET_OF = dict(zip(meta['smear'], meta['set']))
print(f"\n{len(TEST)} held-out smears · {sum(len(v) for v in GT.values())} cells "
      f"({sum(i.cls == 1 for v in GT.values() for i in v)} elongated)")


## 5 · Run the pipeline on the held-out smears

Two numbers, and they answer different questions.

- **Detection** — class-agnostic F1 at mask IoU 0.50, greedy best-IoU matching, one prediction per
  cell. This is step 3's metric and should reproduce step 3's `yolo-finetuned` row.
- **Classification on matched cells** — of the cells the pipeline actually found, how many did it
  label correctly. Cells that were never detected are not counted here; they are counted in the
  detection recall, which is where a miss belongs.

The **per-smear sickle percentage** at the end is what the pipeline is actually for, and it is the
one number that carries both errors at once.


In [ ]:
def match(gt, pr, thr=IOU_T):
    """greedy best-IoU matching -> [(gt_index, pred_index, iou)]"""
    if not gt or not pr: return []
    M = np.array([[iou(g, p) for p in pr] for g in gt])
    out = []
    while True:
        k = int(M.argmax()); g, p = divmod(k, M.shape[1])
        if M[g, p] < thr: break
        out.append((g, p, float(M[g, p])))
        M[g, :] = -1; M[:, p] = -1
    return out


PRED, PAIRS, per_smear = {}, [], []
t0 = time.time()
for sm in TEST:
    insts, _, _, cells = run_pipeline(SRC[sm])
    PRED[sm] = insts
    mm = match(GT[sm], insts)
    for g, p, i in mm:
        PAIRS.append(dict(smear=sm, set=SET_OF[sm], iou=i, y=GT[sm][g].cls,
                          pred=insts[p].cls, p_sickle=insts[p].score))
    gt_k = sum(i.cls == 1 for i in GT[sm])
    pr_k = sum(i.cls == 1 for i in insts)
    per_smear.append(dict(smear=sm, set=SET_OF[sm], gt=len(GT[sm]), pred=len(insts), tp=len(mm),
                          sum_iou=sum(i for _, _, i in mm), gt_sickle=gt_k, pred_sickle=pr_k,
                          gt_pct=100 * gt_k / max(len(GT[sm]), 1),
                          pred_pct=100 * pr_k / max(len(insts), 1)))
    print(f'  {sm:<24s} {len(insts):>4d} detected / {len(GT[sm]):>4d} gt · {len(mm):>4d} matched · '
          f'sickle {pr_k:>3d} vs {gt_k:>3d}')
ps = pd.DataFrame(per_smear); pairs = pd.DataFrame(PAIRS)
print(f'{time.time() - t0:.0f}s for {len(TEST)} smears')

# PR/RC, not P/R -- in the SAM notebook `P` is the predictor object and rebinding it here would
# break run_pipeline for every cell after this one
tp, ngt, npr = ps['tp'].sum(), ps['gt'].sum(), ps['pred'].sum()
PR, RC = tp / max(npr, 1), tp / max(ngt, 1)
DET = dict(F1=2 * PR * RC / max(PR + RC, 1e-9), precision=PR, recall=RC,
           mIoU=ps['sum_iou'].sum() / max(tp, 1), pred=int(npr), gt=int(ngt))
print('\ndetection (class-agnostic, IoU 0.50): ' +
      str({k: round(v, 4) if isinstance(v, float) else v for k, v in DET.items()}))

y, yh = pairs['y'].to_numpy(), pairs['pred'].to_numpy()
CM = np.zeros((2, 2), int)
for a, b in zip(y, yh): CM[a, b] += 1
f1 = []
for c in (0, 1):
    p = CM[c, c] / max(CM[:, c].sum(), 1); r = CM[c, c] / max(CM[c].sum(), 1)
    f1.append(2 * p * r / max(p + r, 1e-9))
CLS = dict(n_matched=len(pairs), accuracy=float((y == yh).mean()), macro_f1=float(np.mean(f1)),
           f1_circular=float(f1[0]), f1_elongated=float(f1[1]),
           sickle_recall=float(CM[1, 1] / max(CM[1].sum(), 1)))
print('classification on matched cells:  ' + str({k: round(v, 4) if isinstance(v, float) else v
                                                  for k, v in CLS.items()}))
print(f'confusion (rows = gt circular/elongated):\n{CM}')

END = dict(mae_sickle_pct=float((ps['pred_pct'] - ps['gt_pct']).abs().mean()),
           bias_sickle_pct=float((ps['pred_pct'] - ps['gt_pct']).mean()))
print(f"\nper-smear sickle %: mean absolute error {END['mae_sickle_pct']:.1f} points, "
      f"bias {END['bias_sickle_pct']:+.1f}")
print(ps[['smear', 'set', 'gt', 'pred', 'tp', 'gt_pct', 'pred_pct']].round(1).to_string(index=False))


## 6 · Pictures and files


In [ ]:
COL = {0: np.array([0.30, 0.45, 0.69]), 1: np.array([0.77, 0.31, 0.32])}   # circular / elongated


def overlay(ax, canvas, insts, title):
    im = cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB).astype(float) / 255
    ov = im.copy()
    for ins in insts:
        sl = (slice(ins.y0, ins.y1), slice(ins.x0, ins.x1))
        ov[sl][ins.m] = 0.35 * im[sl][ins.m] + 0.65 * COL[ins.cls]
    ax.imshow(ov); ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values(): s.set_visible(False)
    ax.set_title(title, fontsize=8)


SHOW = TEST[:N_SHOW]
fig, axs = plt.subplots(len(SHOW), 3, figsize=(9.5, 3.1 * len(SHOW)), squeeze=False)
for r, sm in enumerate(SHOW):
    canvas = to_canvas(cv2.imread(SRC[sm]), cv2.INTER_AREA)
    k = lambda ins: sum(i.cls == 1 for i in ins)
    overlay(axs[r][0], canvas, [], 'source' if r == 0 else '')
    overlay(axs[r][1], canvas, GT[sm], f'ground truth\n{len(GT[sm])} cells, {k(GT[sm])} sickle'
            if r == 0 else f'{len(GT[sm])} cells, {k(GT[sm])} sickle')
    overlay(axs[r][2], canvas, PRED[sm], f'pipeline\n{len(PRED[sm])} cells, {k(PRED[sm])} sickle'
            if r == 0 else f'{len(PRED[sm])} cells, {k(PRED[sm])} sickle')
    axs[r][0].set_ylabel(f'{sm}\n{SET_OF[sm]}', fontsize=7)
fig.suptitle('Step 4 · YOLO-seg (fine-tuned) → crop → ' + CLF_NAME +
             '   ·   blue = circular, red = elongated', fontsize=11)
fig.tight_layout(); fig.savefig(f'{FIG}/step4_yolo_examples.png', dpi=140, bbox_inches='tight')
plt.show()

fig, ax = plt.subplots(1, 3, figsize=(15, 4.2))
ax[0].bar(['F1', 'precision', 'recall', 'mIoU'],
          [DET['F1'], DET['precision'], DET['recall'], DET['mIoU']], color='#2a6')
ax[0].set_ylim(0, 1.02); ax[0].grid(axis='y', alpha=.3); ax[0].set_title('detection @ IoU 0.50')
ax[1].imshow(CM, cmap='Blues')
for i in range(2):
    for j in range(2):
        ax[1].text(j, i, CM[i, j], ha='center', va='center',
                   color='white' if CM[i, j] > CM.max() / 2 else 'black')
ax[1].set_xticks([0, 1], CLASSES); ax[1].set_yticks([0, 1], CLASSES)
ax[1].set_xlabel('predicted'); ax[1].set_ylabel('ground truth')
ax[1].set_title(f"classification on {CLS['n_matched']} matched cells\nacc {CLS['accuracy']:.3f} · "
                f"macro-F1 {CLS['macro_f1']:.3f}")
lim = max(ps['gt_pct'].max(), ps['pred_pct'].max()) * 1.15 + 1
ax[2].plot([0, lim], [0, lim], '--', c='grey', lw=1)
for s, g in ps.groupby('set'):
    ax[2].scatter(g['gt_pct'], g['pred_pct'], label=s, s=45)
ax[2].set_xlim(0, lim); ax[2].set_ylim(0, lim); ax[2].grid(alpha=.3); ax[2].legend()
ax[2].set_xlabel('ground-truth sickle %'); ax[2].set_ylabel('pipeline sickle %')
ax[2].set_title(f"per-smear sickle burden\nMAE {END['mae_sickle_pct']:.1f} points")
fig.suptitle(f'Step 4 · YOLO pipeline on {len(TEST)} held-out Cuba smears', fontsize=11)
fig.tight_layout(); fig.savefig(f'{FIG}/step4_yolo_scores.png', dpi=140, bbox_inches='tight')
plt.show()

ps.to_csv(f'{OUT}/step4_yolo_per_smear.csv', index=False)
pairs.to_csv(f'{OUT}/step4_yolo_matched_cells.csv', index=False)
json.dump(dict(segmenter='yolo11-seg finetuned', seg_ckpt=str(SEG_CKPT), classifier=CLF_NAME,
               clf_ckpt=str(CLF_CKPT), threshold=CLF_THR, crop=CROP, pad=PAD, canvas=WORK,
               iou_threshold=IOU_T, n_test_smears=len(TEST), detection=DET,
               classification_on_matched=CLS, end_to_end=END),
          open(f'{OUT}/step4_yolo_summary.json', 'w'), indent=2)

BUNDLE = f'{OUT}/scd_final_step4_yolo_downloads.zip'
with zipfile.ZipFile(BUNDLE, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in sorted(glob.glob(f'{FIG}/*.png')): z.write(f, f'figures/{os.path.basename(f)}')
    for f in ('step4_yolo_per_smear.csv', 'step4_yolo_matched_cells.csv',
              'step4_yolo_summary.json'): z.write(f'{OUT}/{f}', f)
print(f'{BUNDLE}  ({os.path.getsize(BUNDLE) / 1e6:.1f} MB)')


## 7 · Use it on any smear

This is the actual product: one image path in, a per-cell table and a sickle count out. It needs no
ground truth and no split — it runs on a smear the pipeline has never seen, from either population.
Point `IMAGE` at anything.


In [ ]:
IMAGE = os.environ.get('SCD_DEMO_IMAGE', SRC[TEST[0]])

insts, canvas, src, cells = run_pipeline(IMAGE)
r = report(cells, os.path.basename(IMAGE))
print(cells.head(10).round(3).to_string(index=False))
cells.to_csv(f'{OUT}/demo_cells.csv', index=False)

fig, ax = plt.subplots(figsize=(7, 7))
overlay(ax, canvas, insts, f"{os.path.basename(IMAGE)} · {r['cells']} cells · "
                           f"{r['sickle']} sickle ({r['sickle_pct']:.1f}%)")
fig.tight_layout(); fig.savefig(f'{FIG}/demo.png', dpi=140, bbox_inches='tight'); plt.show()
print(f'-> {OUT}/demo_cells.csv')
